In [ ]:
links_of_github=[
    'https://github.com/orgs/Seeed-Studio/repositories',
    'https://github.com/orgs/greatscottgadgets/repositories',
    'https://github.com/orgs/OSHPark/repositories',
    'https://github.com/orgs/pimoroni/repositories',
    'https://github.com/Arduino'
]

# Scrap adafruit

In [5]:
import requests
from urllib.parse import urlparse, parse_qs, urlencode, urlunparse
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import csv
import os
import re

STOP_TEXT = "No more repositories. Visit a lower page."

import time
from requests.adapters import HTTPAdapter, Retry


def get_all_page_links(base_url, max_pages=None, delay_sec=1.0):
    """
    Get all GitHub org repo listing pages until STOP_TEXT is found or max_pages is reached.
    Adds delay between requests to avoid 429 Too Many Requests.
    """
    session = requests.Session()
    session.headers["User-Agent"] = "Mozilla/5.0"

    retries = Retry(
        total=5,
        backoff_factor=2,  # exponential backoff
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"]
    )
    session.mount("https://", HTTPAdapter(max_retries=retries))

    parsed = urlparse(base_url)
    query = parse_qs(parsed.query)

    page_num = 1
    page_links = []

    while True:
        if max_pages and page_num > max_pages:
            break

        query["page"] = [str(page_num)]
        new_query = urlencode(query, doseq=True)
        page_url = urlunparse(parsed._replace(query=new_query))

        resp = session.get(page_url)
        resp.raise_for_status()

        if STOP_TEXT in resp.text:
            break

        page_links.append(page_url)
        print(f"Collected page {page_num}: {page_url}")

        page_num += 1
        time.sleep(delay_sec)  # polite pause

    return page_links


def get_repo_items_headless(page_url, wait_sec=15):
    # Chrome headless options
    options = webdriver.ChromeOptions()
    options.add_argument("--headless=new")  # headless mode
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--window-size=1920,1080")

    driver = webdriver.Chrome(options=options)
    try:
        driver.get(page_url)

        # Wait for the container whose id ends with -list-view-container
        container = WebDriverWait(driver, wait_sec).until(
            EC.presence_of_element_located(
                (By.CSS_SELECTOR, '[id$="-list-view-container"]')
            )
        )

        # Find all list items whose id contains -list-view-node-
        items = container.find_elements(By.CSS_SELECTOR, 'li[id*="-list-view-node-"]')

        repos = []
        for li in items:
            name = li.find_element(By.CSS_SELECTOR, "h4 a span").text.strip()
            href = li.find_element(By.CSS_SELECTOR, "h4 a").get_attribute("href")
            if href and href.startswith("/"):
                href = "https://github.com" + href
            repos.append({"name": name, "url": href})

        return repos
    finally:
        driver.quit()


def get_all_org_repo_links(base_url):
    pages = get_all_page_links(base_url)
    print(f"Found {len(pages)} pages: {pages}")

    all_repos = []
    for page in pages:
        repo_links = get_repo_items_headless(page)  # list of dicts
        all_repos.extend(repo_links)

    # Deduplicate by repo["url"]
    seen = set()
    unique_repos = []
    for repo in all_repos:
        if repo["url"] not in seen:
            seen.add(repo["url"])
            unique_repos.append(repo)

    print(f"\nTotal repositories found: {len(unique_repos)}")
    return unique_repos


def save_repo_list_to_csv(repo_list, filename="repo_list.csv"):
    """
    Save a list of repo URLs or dicts to a CSV file.
    If the list contains dicts with 'name' and 'url', both columns are saved.
    If the list contains strings, only 'url' is saved.
    """
    with open(filename, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        
        if repo_list and isinstance(repo_list[0], dict):
            writer.writerow(["name", "url"])
            for repo in repo_list:
                writer.writerow([repo.get("name", ""), repo.get("url", "")])
        else:
            writer.writerow(["url"])
            for url in repo_list:
                writer.writerow([url])

    print(f"Saved {len(repo_list)} entries to {filename}")


# Example

In [12]:
base_url = links_of_github[2]
# Extract org name safely
match = re.search(r'/orgs/([^/]+)/', base_url)
if not match:
    raise ValueError(f"Could not extract org name from URL: {base_url}")
source_name = match.group(1)

# Get repo list
repo_list = get_all_org_repo_links(base_url)

# Build CSV path safely
csv_input = os.path.join(
    r"C:\Users\Taiting\Desktop\yolo_training_24",
    f"{source_name}_repos.csv"
)

# Save
save_repo_list_to_csv(repo_list, csv_input)
print(f"Saved {len(repo_list)} repos to {csv_input}")

Collected page 1: https://github.com/orgs/greatscottgadgets/repositories?page=1
Collected page 2: https://github.com/orgs/greatscottgadgets/repositories?page=2
Found 2 pages: ['https://github.com/orgs/greatscottgadgets/repositories?page=1', 'https://github.com/orgs/greatscottgadgets/repositories?page=2']

Total repositories found: 48
Saved 48 entries to C:\Users\Taiting\Desktop\yolo_training_24\greatscottgadgets_repos.csv
Saved 48 repos to C:\Users\Taiting\Desktop\yolo_training_24\greatscottgadgets_repos.csv


# Download Github Repo

In [ ]:
import csv
import os
import re
import time
import requests
from urllib.parse import urlparse

# ---------- Helpers ----------
def _session_with_retries(total=3, backoff=0.5, timeout=60):
    s = requests.Session()
    s.headers.update({
        "User-Agent": "github-zip-downloader/1.0",
        "Accept": "application/vnd.github+json",
    })
    from requests.adapters import HTTPAdapter, Retry
    retries = Retry(
        total=total,
        backoff_factor=backoff,
        status_forcelist=(429, 500, 502, 503, 504),
        allowed_methods=frozenset(["GET", "HEAD"])
    )
    s.mount("http://", HTTPAdapter(max_retries=retries))
    s.mount("https://", HTTPAdapter(max_retries=retries))
    s.request_timeout = timeout
    return s

def _sanitize_name(name: str) -> str:
    # Remove characters not suitable for folder names
    return re.sub(r'[\\/:*?"<>|]+', "_", name).strip()

def _parse_owner_repo(repo_url: str):
    """
    Accepts https://github.com/Owner/Repo (extras like /issues are OK; they get trimmed)
    Returns (owner, repo) or (None, None)
    """
    try:
        p = urlparse(repo_url)
        if p.netloc.lower() != "github.com":
            return None, None
        parts = [x for x in p.path.split("/") if x]
        if len(parts) < 2:
            return None, None
        owner, repo = parts[0], parts[1]
        # strip .git if present
        repo = repo[:-4] if repo.endswith(".git") else repo
        return owner, repo
    except Exception:
        return None, None

def _get_default_branch(owner, repo, session, token=None):
    """
    Use GitHub API to get default branch. Falls back to 'main' then 'master'.
    """
    if token:
        session.headers["Authorization"] = f"Bearer {token}"
    api = f"https://api.github.com/repos/{owner}/{repo}"
    try:
        r = session.get(api, timeout=session.request_timeout)
        if r.status_code == 200:
            data = r.json()
            if "default_branch" in data and data["default_branch"]:
                return data["default_branch"]
    except requests.RequestException:
        pass
    return None  # let caller try fallbacks

def _download_archive(owner, repo, branch, dest_zip_path, session):
    """
    Download https://github.com/{owner}/{repo}/archive/refs/heads/{branch}.zip
    Returns True/False
    """
    url = f"https://github.com/{owner}/{repo}/archive/refs/heads/{branch}.zip"
    try:
        with session.get(url, stream=True, timeout=session.request_timeout) as r:
            if r.status_code >= 400:
                return False
            os.makedirs(os.path.dirname(dest_zip_path), exist_ok=True)
            with open(dest_zip_path, "wb") as f:
                for chunk in r.iter_content(chunk_size=1024 * 256):
                    if chunk:
                        f.write(chunk)
        return True
    except requests.RequestException:
        return False


def download_repos_from_csv(csv_path, dest_folder, token=None, delay_sec=0.0, skip_existing=True):
    """
    Read CSV with header: name,url
    For each row:
      - create folder: {dest_folder}/{name}
      - download ZIP of default branch (fallback to main/master)
      - save as {dest_folder}/{name}/{name}.zip

    Returns:
        (success_list, fail_list)
    """
    session = _session_with_retries()

    successes, failures = [], []

    # Read all rows first so we know total for progress display
    with open(csv_path, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        if "name" not in reader.fieldnames or "url" not in reader.fieldnames:
            raise ValueError("CSV must have headers: name,url")
        rows = list(reader)

    total = len(rows)

    for idx, row in enumerate(rows, start=1):
        raw_name = (row.get("name") or "").strip()
        url = (row.get("url") or "").strip()
        print(f"[{idx}/{total}] {raw_name or '(missing name)'} — {url}")

        if not raw_name or not url:
            failures.append({"name": raw_name, "url": url, "reason": "missing name or url"})
            print("  -> SKIP (missing name/url)")
            continue

        name = _sanitize_name(raw_name)
        owner, repo = _parse_owner_repo(url)
        if not owner or not repo:
            failures.append({"name": name, "url": url, "reason": "not a valid GitHub repo URL"})
            print("  -> FAILED (invalid repo URL)")
            continue

        repo_dir = os.path.join(dest_folder, name)
        zip_path = os.path.join(repo_dir, f"{name}.zip")

        if skip_existing and os.path.isfile(zip_path):
            print("  -> Skipped (zip already exists)")
            successes.append({"name": name, "url": url, "zip_path": zip_path})
            continue

        # Determine default branch; fall back to main/master
        default_branch = _get_default_branch(owner, repo, session, token=token)
        branches_to_try = [b for b in [default_branch, "main", "master"] if b]

        ok = False
        tried = []
        os.makedirs(repo_dir, exist_ok=True)

        for br in branches_to_try:
            tried.append(br)
            print(f"  -> Trying branch: {br}")
            if _download_archive(owner, repo, br, zip_path, session):
                ok = True
                print(f"  -> Downloaded to: {zip_path}")
                break

        if ok:
            successes.append({"name": name, "url": url, "zip_path": zip_path})
        else:
            reason = f"failed to download branches: {', '.join(tried) if tried else 'none'}"
            failures.append({"name": name, "url": url, "reason": reason})
            print(f"  -> FAILED ({reason})")

        if delay_sec:
            time.sleep(delay_sec)

    print(f"\nDone. Downloaded: {len(successes)}, Failed: {len(failures)}")
    return successes, failures


# Example

In [ ]:
# ---------- Example usage ----------

csv_input = "/Users/taitinglu/Desktop/IMG2SCH/Seeed-Studio_repos.csv"  # header: name,url
dest = "/Users/taitinglu/Desktop/IMG2SCH/Seeed-Studio_repos"
success, fail = download_repos_from_csv(csv_input, dest, token=None, delay_sec=0.2)

print(f"Downloaded: {len(success)}, Failed: {len(fail)}")
if fail:
    for item in fail[:10]:
        print("FAIL:", item)

# UnZip Repo

In [ ]:
import csv
import os
import zipfile
from pathlib import Path
import shutil

def _safe_extract_zip(zip_path: Path, extract_to: Path):
    extract_to = extract_to.resolve()
    with zipfile.ZipFile(zip_path, "r") as zf:
        for m in zf.infolist():
            target = (extract_to / m.filename).resolve()
            if not str(target).startswith(str(extract_to)):
                raise RuntimeError(f"Unsafe path in zip: {m.filename}")
        zf.extractall(extract_to)

def _flatten_single_top_dir(root: Path):
    """
    If extraction creates a single top-level folder, move its contents up one level.
    """
    entries = [p for p in root.iterdir() if p.exists()]
    if len(entries) == 1 and entries[0].is_dir():
        top = entries[0]
        for p in top.iterdir():
            shutil.move(str(p), str(root / p.name))
        shutil.rmtree(top, ignore_errors=True)

def unzip_repos_from_csv(
    csv_path: str,
    dest_folder: str,
    *,
    skip_existing: bool = True,
    delete_zip_after: bool = False,
    flatten_top_dir: bool = True,
):
    """
    For each row in CSV (headers: name,url), look in {dest}/{name}/ for any .zip
    and extract it into {dest}/{name}/. Shows progress and supports skipping,
    deleting zips after extraction, and flattening the top-level folder.
    """
    csv_path = Path(csv_path)
    dest_folder = Path(dest_folder)

    # read rows
    with csv_path.open(newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        if "name" not in reader.fieldnames or "url" not in reader.fieldnames:
            raise ValueError("CSV must have headers: name,url")
        rows = list(reader)

    successes, failures = [], []
    total = len(rows)

    for idx, row in enumerate(rows, start=1):
        name = (row.get("name") or "").strip()
        repo_dir = dest_folder / name
        print(f"[{idx}/{total}] {name or '(missing name)'}")

        if not name:
            failures.append({"name": name, "reason": "missing name"})
            print("  -> FAILED (missing name)")
            continue

        if not repo_dir.is_dir():
            failures.append({"name": name, "reason": f"folder not found: {repo_dir}"})
            print(f"  -> FAILED (folder missing: {repo_dir})")
            continue

        # find any zip inside the repo folder
        zips = list(repo_dir.glob("*.zip"))
        if not zips:
            failures.append({"name": name, "reason": "zip not found"})
            print("  -> FAILED (zip not found)")
            continue

        # if multiple, pick the largest (usually the actual archive)
        zip_path = max(zips, key=lambda p: p.stat().st_size)

        # skip if already extracted (non-empty dir) and skip_existing
        if skip_existing:
            try:
                if any(repo_dir.iterdir()):
                    # if the only thing is the zip, still extract
                    only_zip = (len(list(repo_dir.iterdir())) == 1 and list(repo_dir.iterdir())[0] == zip_path)
                    if not only_zip:
                        print("  -> Skipped (already extracted)")
                        successes.append({"name": name, "zip_path": str(zip_path), "extract_dir": str(repo_dir)})
                        continue
            except FileNotFoundError:
                pass

        try:
            _safe_extract_zip(zip_path, repo_dir)
            if flatten_top_dir:
                _flatten_single_top_dir(repo_dir)
            if delete_zip_after:
                try:
                    zip_path.unlink(missing_ok=True)
                except TypeError:
                    # for older Python
                    if zip_path.exists():
                        zip_path.unlink()
            print(f"  -> Extracted to {repo_dir}")
            successes.append({"name": name, "zip_path": str(zip_path), "extract_dir": str(repo_dir)})
        except zipfile.BadZipFile:
            failures.append({"name": name, "reason": "bad zip", "zip_path": str(zip_path)})
            print("  -> FAILED (bad zip)")
        except Exception as e:
            failures.append({"name": name, "reason": str(e), "zip_path": str(zip_path)})
            print(f"  -> FAILED ({e})")

    print(f"\nDone. Unzipped {len(successes)} / {total}. Failures: {len(failures)}")
    return successes, failures



# Example

In [ ]:
source_name = __import__('re').search(r'/orgs/([^/]+)/', base_url).group(1)
csv_input = "/Users/taitinglu/Desktop/IMG2SCH/"+source_name+"_repos.csv"  # header: name,url
dest = "/Users/taitinglu/Desktop/IMG2SCH/"+source_name+"_repos"

unzip_repos_from_csv(
    csv_input,
    dest,
    skip_existing=True,
    delete_zip_after=False,   # set True to remove zips after extraction
    flatten_top_dir=True
)


# Copy Sch Brd to Subfolder

In [ ]:
import csv
from pathlib import Path
import shutil

def copy_sch_brd_from_csv(csv_path: str, dest_folder: str):
    """
    CSV: header 'name,url'
    For each row, scans {dest_folder}/{name}/ recursively and COPIES:
      - *.sch -> {dest_folder}/sch/
      - *.brd -> {dest_folder}/brd/

    Returns:
        dict: {"sch": int, "brd": int, "miss": int}
    """
    dest_root = Path(dest_folder)
    out_sch = dest_root / "sch"
    out_brd = dest_root / "brd"
    out_sch.mkdir(parents=True, exist_ok=True)
    out_brd.mkdir(parents=True, exist_ok=True)

    # load rows
    with open(csv_path, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        if "name" not in reader.fieldnames or "url" not in reader.fieldnames:
            raise ValueError("CSV must have headers: name,url")
        rows = list(reader)

    counts = {"sch": 0, "brd": 0, "miss": 0}
    total = len(rows)

    def _unique_dest(p: Path) -> Path:
        """Avoid overwriting: add _1, _2, ... suffix if needed."""
        if not p.exists():
            return p
        stem, ext = p.stem, p.suffix
        i = 1
        while True:
            cand = p.with_name(f"{stem}_{i}{ext}")
            if not cand.exists():
                return cand
            i += 1

    for idx, row in enumerate(rows, start=1):
        name = (row.get("name") or "").strip()
        repo_dir = dest_root / name
        print(f"[{idx}/{total}] {name or '(missing name)'}")

        if not name or not repo_dir.is_dir():
            print(f"  -> SKIP (repo folder missing: {repo_dir})")
            counts["miss"] += 1
            continue

        copied_any = False
        for path in repo_dir.rglob("*"):
            if not path.is_file():
                continue
            ext = path.suffix.lower()
            if ext not in (".sch", ".brd"):
                continue

            target_dir = out_sch if ext == ".sch" else out_brd
            dest_path = _unique_dest(target_dir / path.name)

            # ensure parent exists, then copy
            dest_path.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(str(path), str(dest_path))
            counts[ext.lstrip(".")] += 1
            copied_any = True
            print(f"  -> Copied: {path.relative_to(dest_root)}  ->  {dest_path.relative_to(dest_root)}")

        if not copied_any:
            print("  -> No .sch/.brd found")

    print(f"\nDone. Copied {counts['sch']} .sch and {counts['brd']} .brd files. "
          f"Missing/empty repos: {counts['miss']}")
    return counts



# Example

In [ ]:

copy_sch_brd_from_csv(csv_input, dest)